In [1]:
# Cell 1: installs (Kaggle often has these, but let's be explicit)
!pip -q install decord

import os, sys, math, random, time
from pathlib import Path
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import cv2
from decord import VideoReader, cpu

from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 86.2 MB/s eta 0:00:00:00:01:01
PyTorch: 2.6.0+cu124
CUDA available: True
GPUs: 2


In [2]:
# Cell 2: configuration

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True  # speed for fixed-size inputs

seed_all(42)

# >>> YOUR DATA ROOTS (matches your screenshot/path)
DATA_ROOT = "/kaggle/input/rwf2000/RWF-2000"  # contains train/ and val/
TRAIN_DIR = f"{DATA_ROOT}/train"
VAL_DIR   = f"{DATA_ROOT}/val"

# Hyperparameters (tweak as needed)
CLIP_LEN   = 16          # try 32 later for a boost (slower)
SIZE       = 112         # try 128/160 if GPU allows
BATCH_SIZE = 20          # 2× T4 should handle this; lower if OOM
EPOCHS     = 5
LR         = 1e-4
WARMUP_STEPS = 400
NUM_WORKERS  = 4

OUT_DIR = "/kaggle/working/checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"Using device={device}, GPUs={n_gpus}")


Using device=cuda, GPUs=2


In [3]:
# Cell 3: dataset + preprocessing

KINETICS_MEAN = [0.43216, 0.394666, 0.37645]
KINETICS_STD  = [0.22803, 0.22145, 0.216989]

def _indices(n, T, training=True):
    if n <= 0: return np.zeros((T,), dtype=np.int64)
    if n <= T: return np.linspace(0, n-1, T).astype(np.int64)
    if training:
        stride = max(1, n // T)
        start  = random.randint(0, max(n - stride*T, 0))
        return (start + np.arange(T) * stride).clip(0, n-1).astype(np.int64)
    return np.linspace(0, n-1, T).astype(np.int64)

def _jitter_color_rgb(img_rgb, b=0.10, c=0.10, s=0.05, h=0.02):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[...,1] *= (1.0 + np.random.uniform(-s, s))
    hsv[...,0]  = (hsv[...,0] + (h*180.0*np.random.uniform(-1,1))) % 180.0
    img2 = cv2.cvtColor(np.clip(hsv,0,255).astype(np.uint8), cv2.COLOR_HSV2RGB)
    alpha = 1.0 + np.random.uniform(-c, c)
    beta  = 255.0 * np.random.uniform(-b, b)
    img2  = np.clip(alpha*img2 + beta, 0, 255).astype(np.uint8)
    return img2

def _augment_clip(frames):
    # Apply the SAME transform to every frame (temporal consistency)
    if random.random() < 0.5:  # flip
        frames = frames[:, :, ::-1, :]
    if random.random() < 0.5:  # mild color jitter
        frames = np.stack([_jitter_color_rgb(fr) for fr in frames], axis=0)
    if random.random() < 0.10: # slight blur
        k = random.choice([3,5])
        frames = np.stack([cv2.GaussianBlur(fr,(k,k),0) for fr in frames], axis=0)
    if random.random() < 0.05: # occasional grayscale
        g = np.stack([cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY) for fr in frames], axis=0)
        frames = np.repeat(g[...,None], 3, axis=-1).astype(np.uint8)
    return frames

class RWF2000(Dataset):
    """
    Expects:
      root/
        train/{Fight,NonFight}/*.avi
        val/{Fight,NonFight}/*.avi
    """
    def __init__(self, root, split="train", clip_len=16, size=112, training=True, augment=True):
        self.root = Path(root)
        self.split = split
        self.clip_len = clip_len
        self.size = size
        self.training = training
        self.augment = (augment and training)

        self.samples = []
        for label_name in ["Fight", "NonFight"]:
            y = 1 if label_name.lower().startswith("fight") else 0
            d = self.root / split / label_name
            if not d.exists(): 
                continue
            for p in d.glob("*.*"):
                if p.suffix.lower() in {".avi",".mp4",".mkv",".mov"}:
                    self.samples.append((str(p), y))

        if not self.samples:
            raise RuntimeError(f"No videos found in {self.root}/{split}/(Fight|NonFight)")

        self.mean = torch.tensor(KINETICS_MEAN).view(3,1,1,1)
        self.std  = torch.tensor(KINETICS_STD).view(3,1,1,1)

    def __len__(self): 
        return len(self.samples)

    def __getitem__(self, i):
        path, y = self.samples[i]
        vr = VideoReader(path, ctx=cpu(0))
        idx = _indices(len(vr), self.clip_len, self.training)
        frames = vr.get_batch(idx).asnumpy()  # (T,H,W,3) RGB uint8

        if self.augment:
            frames = _augment_clip(frames)

        frames = np.stack([cv2.resize(fr, (self.size, self.size), cv2.INTER_LINEAR)
                           for fr in frames], axis=0)  # (T, S, S, 3)

        x = torch.from_numpy(frames).float().div(255.0).permute(3,0,1,2)  # (3,T,S,S)
        x = (x - self.mean) / self.std
        return x, torch.tensor(y, dtype=torch.long)


In [4]:
# Cell 4: dataloaders + a quick peek

train_ds = RWF2000(root=DATA_ROOT, split="train", clip_len=CLIP_LEN, size=SIZE, training=True,  augment=True)
val_ds   = RWF2000(root=DATA_ROOT, split="val",   clip_len=CLIP_LEN, size=SIZE, training=False, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

len_train, len_val = len(train_ds), len(val_ds)
print(f"Train videos: {len_train} | Val videos: {len_val}")

# Peek one batch (shapes)
xb, yb = next(iter(train_loader))
print("Batch video shape:", xb.shape, "(B, C, T, H, W)")
print("Batch labels shape:", yb.shape, "unique labels:", yb.unique())


Train videos: 1600 | Val videos: 400
Batch video shape: torch.Size([20, 3, 16, 112, 112]) (B, C, T, H, W)
Batch labels shape: torch.Size([20]) unique labels: tensor([0, 1])


In [5]:
# Cell 5: model

weights = R2Plus1D_18_Weights.KINETICS400_V1
model = r2plus1d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)

if n_gpus >= 2:
    model = nn.DataParallel(model)  # simple multi-GPU over 2× T4

model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

print("Model ready.")


Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to /root/.cache/torch/hub/checkpoints/r2plus1d_18-91a641e6.pth
100%|██████████| 120M/120M [00:00<00:00, 196MB/s] 


Model ready.


/tmp/ipykernel_37/587672777.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))


In [6]:
# Cell 6: evaluation

@torch.no_grad()
def evaluate(model, loader, device="cuda"):
    model.eval()
    ce = nn.CrossEntropyLoss(reduction="sum")
    total = correct = 0
    tp = tn = fp = fn = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss_sum += ce(logits, y).item()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total   += y.numel()
        tp += ((pred==1) & (y==1)).sum().item()
        tn += ((pred==0) & (y==0)).sum().item()
        fp += ((pred==1) & (y==0)).sum().item()
        fn += ((pred==0) & (y==1)).sum().item()

    acc  = correct / max(1,total)
    prec = tp / max(1,tp+fp)
    rec  = tp / max(1,tp+fn)
    f1   = (2*prec*rec / max(1e-8, prec+rec)) if (prec+rec) > 0 else 0.0
    return loss_sum/max(1,total), acc, prec, rec, f1


In [ ]:
# Cell 7 (with tqdm): training loop + nice progress bars
from tqdm.auto import tqdm

steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * EPOCHS

def cosine_lr(step, warmup=WARMUP_STEPS, total=total_steps):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))

best_f1 = -1.0
global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0; run_total = 0; run_correct = 0

    # progress bar for training batches
    pbar = tqdm(train_loader, total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # LR schedule step
        lr_scale = cosine_lr(global_step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr_scale * LR
        current_lr = optimizer.param_groups[0]["lr"]

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        run_loss   += loss.item() * x.size(0)
        run_total  += y.numel()
        run_correct += (logits.argmax(1) == y).sum().item()
        global_step += 1

        # update bar postfix with running stats
        train_loss_running = run_loss / max(1, run_total)
        train_acc_running  = run_correct / max(1, run_total)
        pbar.set_postfix({
            "lr": f"{current_lr:.2e}",
            "loss": f"{train_loss_running:.3f}",
            "acc": f"{train_acc_running:.3f}"
        })

    # end of epoch: compute epoch train stats
    train_loss = run_loss / max(1, run_total)
    train_acc  = run_correct / max(1, run_total)

    # validation with its own bar
    val_iter = tqdm(val_loader, total=len(val_loader), desc="Validating", leave=False)
    # (optional) you can show a bar while evaluate() runs by re-implementing here;
    # simplest is to call evaluate() directly (no per-batch updates). For a visible bar:
    model.eval()
    ce = nn.CrossEntropyLoss(reduction="sum")
    total = correct = 0
    tp = tn = fp = fn = 0
    val_loss_sum = 0.0
    with torch.no_grad():
        for x, y in val_iter:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            val_loss_sum += ce(logits, y).item()
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total   += y.numel()
            tp += ((pred==1) & (y==1)).sum().item()
            tn += ((pred==0) & (y==0)).sum().item()
            fp += ((pred==1) & (y==0)).sum().item()
            fn += ((pred==0) & (y==1)).sum().item()
    val_acc  = correct / max(1,total)
    val_prec = tp / max(1,tp+fp)
    val_rec  = tp / max(1,tp+fn)
    val_f1   = (2*val_prec*val_rec / max(1e-8, val_prec+val_rec)) if (val_prec+val_rec) > 0 else 0.0
    val_loss = val_loss_sum / max(1,total)

    # print a single clean line per epoch
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f} P {val_prec:.3f} R {val_rec:.3f} F1 {val_f1:.3f}")

    # save best checkpoint by F1
    if val_f1 > best_f1:
        best_f1 = val_f1
        ckpt = os.path.join(OUT_DIR, "best_r2p1d18.pt")
        torch.save((model.module if isinstance(model, nn.DataParallel) else model).state_dict(), ckpt)
        print(f"  -> Saved BEST checkpoint to {ckpt} (F1={best_f1:.3f})")


Epoch 1/5:   0%|          | 0/80 [00:00<?, ?it/s]

/tmp/ipykernel_37/222095861.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


In [8]:
# Cell 8: optional reload best model and evaluate again

ckpt_path = os.path.join(OUT_DIR, "best_r2p1d18.pt")
if os.path.exists(ckpt_path):
    print("Reloading best checkpoint:", ckpt_path)
    # rebuild a fresh model to ensure state_dict is clean
    fresh = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
    fresh.fc = nn.Linear(fresh.fc.in_features, 2)
    fresh.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    if n_gpus >= 2: fresh = nn.DataParallel(fresh)
    fresh = fresh.to(device)
    vloss, vacc, vprec, vrec, vf1 = evaluate(fresh, val_loader, device)
    print(f"[Best@Reload] val loss {vloss:.4f} acc {vacc:.3f} P {vprec:.3f} R {vrec:.3f} F1 {vf1:.3f}")
else:
    print("No checkpoint found yet.")


Reloading best checkpoint: /kaggle/working/checkpoints/best_r2p1d18.pt
[Best@Reload] val loss 0.2958 acc 0.905 P 0.862 R 0.965 F1 0.910


In [14]:
import os
import subprocess
from IPython.display import FileLink, display

def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))


In [15]:
download_file('/kaggle/working/checkpoints/best_r2p1d18.pt', 'out')

/kaggle/working/out.zip